In [2]:
import nltk, warnings
warnings.filterwarnings('ignore')
for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'movie_reviews']:
  nltk.download(pkg, quiet = True)
print('NLP environment ready!')

NLP environment ready!


In [3]:
from nltk.corpus import movie_reviews
docs, labels = [], []
for cat in movie_reviews.categories():
  for fid in movie_reviews.fileids(cat):
    docs.append(movie_reviews.raw(fid))
    labels.append(1 if cat == 'pos' else 0)
print('Documents:', len(docs), '| positive:', sum(labels), '| negative:', len(labels) - sum(labels))
print('First review(first 60 chars):', repr(docs[0][:60]))

Documents: 2000 | positive: 1000 | negative: 1000
First review(first 60 chars): 'plot : two teen couples go to a church party , drink and the'


In [4]:
import spacy
from collections import Counter
nlp = spacy.load('en_core_web_sm')
article_news = ('Sundar Pichai announched on Monday that Google will invest '
                '$2 billion in India by 2027. The plan was discussed in New Delhi '
                'with officials from the Ministry of Electronics, and Reliance '
                'Industries confirmed a partnership last week.')
doc = nlp(article_news)
for ent in doc.ents:
  print(f' {ent.text:<28} {ent.label_}')
print('Entity counts:', dict(Counter(e.label_ for e in doc.ents)))
from spacy import displacy
displacy.render(doc,style='ent', jupyter=True)

 Sundar Pichai                PERSON
 Monday                       DATE
 Google                       ORG
 $2 billion                   MONEY
 India                        GPE
 2027                         DATE
 New Delhi                    GPE
 the Ministry of Electronics  ORG
 Reliance Industries          ORG
 last week                    DATE
Entity counts: {'PERSON': 1, 'DATE': 3, 'ORG': 3, 'MONEY': 1, 'GPE': 2}


In [5]:
import re
text4 = ('Priya Sharma completed B.Sc. Data Science at KRMU in 2024. '
         'Contact her at priya.sharma@example.com or +91 9876543210 '
         'before 15/08/2026 for the machine learning internship position.')
d4 = nlp(text4)
print('Noun chunks:', [c.text for c in d4.noun_chunks])
emails = re.findall(r'[\w.+-]+@[\w-]+\.[\w.]+', text4)
dates = re.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\b\d{4}\b', text4)
phones = re.findall(r'(?:\+91[- ]?)?\d{10}', text4)
print('Structured record:', {'emails': emails, 'dates': dates, 'phones': phones})


Noun chunks: ['Priya Sharma', 'B.Sc', 'Data Science', 'KRMU', 'her', 'the machine', 'internship position']
Structured record: {'emails': ['priya.sharma@example.com'], 'dates': ['2024', '15/08/2026'], 'phones': ['+91 9876543210']}


In [7]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 98.9 MB/s  0:00:00


In [1]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim import corpora
from gensim.models import LdaModel
stop = set(stopwords.words('english'))
tok_docs = [[w for w in word_tokenize(d.lower()) if w.isalpha() and w not in stop and len(w) > 3]
            for d in docs[:400]]
dic = corpora.Dictionary(tok_docs)
dic.filter_extremes(no_below=5, no_above=0.5)
bow = [dic.doc2bow(t) for t in tok_docs]
lda = LdaModel(bow, num_topics=4, id2word=dic, passes=8, random_state=42)
for t in range(4):
    print(f'Topic {t}:', lda.print_topic(t, 6))
mix = sorted(lda.get_document_topics(bow[0]), key=lambda x: -x[1])
print('Document 0 topic mixture:', [(t, round(float(p), 2)) for t, p in mix])

ModuleNotFoundError: No module named 'nltk'

In [9]:
try:
    from transformers import pipeline
    bert = pipeline('sentiment-analysis')
    tricky = ['We had a wonderful time!', 'The movie was not bad at all.', 'Oh great, ANOTHER remake...']
    for s in tricky:
        r = bert(s)[0]
        print(f"{s:<38} -> {r['label']} ({r['score']:.2f})")
    print('Supervised model on sentence 2:', clf.predict(tf.transform([tricky[1]]))[0])
except Exception as e:
    print('Model download requires internet on first run — execute on your machine.  (', type(e).__name__, ')')
    print('Expected: sentence 2 -> POSITIVE (~0.99) from BERT, while the supervised model says 0;')
    print('the sarcastic sentence 3 often fools BOTH models — a shared limitation to record.')

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

We had a wonderful time!               -> POSITIVE (1.00)
The movie was not bad at all.          -> POSITIVE (1.00)
Oh great, ANOTHER remake...            -> POSITIVE (1.00)
Model download requires internet on first run — execute on your machine.  ( NameError )
Expected: sentence 2 -> POSITIVE (~0.99) from BERT, while the supervised model says 0;
the sarcastic sentence 3 often fools BOTH models — a shared limitation to record.


In [10]:
import pandas as pd
rows = [
 ['Test accuracy (this lab)',   '0.82 on movie reviews',        '~0.90+ typical on reviews'],
 ["Handles 'not bad at all'",   'No (predicted negative)',      'Yes (POSITIVE, ~0.99)'],
 ['Prediction speed',           'Milliseconds (linear model)',  'Seconds on CPU'],
 ['Training/setup cost',        'Seconds, 5000 features',       'None (pre-trained, ~260 MB download)'],
 ['Interpretability',           'Full (printable coefficients)','Low (attention not directly readable)'],
 ['Offline capability',         'Yes, tiny footprint',          'Yes after first download; heavy'],
]
print(pd.DataFrame(rows, columns=['Criterion', 'Supervised (TF-IDF+LR)', 'Pre-trained (BERT)']).to_string(index=False))

               Criterion        Supervised (TF-IDF+LR)                    Pre-trained (BERT)
Test accuracy (this lab)         0.82 on movie reviews             ~0.90+ typical on reviews
Handles 'not bad at all'       No (predicted negative)                 Yes (POSITIVE, ~0.99)
        Prediction speed   Milliseconds (linear model)                        Seconds on CPU
     Training/setup cost        Seconds, 5000 features  None (pre-trained, ~260 MB download)
        Interpretability Full (printable coefficients) Low (attention not directly readable)
      Offline capability           Yes, tiny footprint       Yes after first download; heavy
